## 1. Connect to the database
Loads the database address from `.env`, connects with automatic saving, sets the session time zone to UTC (so timestamps are always hashed the same way), and shows how many rows the audit log already has. It should be empty.

In [1]:
import os, sys, json
import pandas as pd
from IPython.display import display, Markdown
import psycopg
from psycopg.types.json import Jsonb

BASE = os.path.expanduser("~/bank-rag-guard")
SRC_DIR = f"{BASE}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)                      # lets Python find our modules in src/

# Read KEY=value lines from .env without printing any secret
for line in open(f"{BASE}/.env", encoding="utf-8"):
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)   # each statement is saved immediately
conn.execute("SET TIME ZONE 'UTC'")                                    # same time zone for writing and checking
print("Connected. Rows already in audit_log:", conn.execute("SELECT count(*) FROM audit_log").fetchone()[0])

Connected. Rows already in audit_log: 0


## 2. Write the audit module (`src/audit.py`)
Defines the three pieces of the hash chain:
- `compute_hash` fingerprints an entry: SHA-256 of the previous fingerprint plus the entry's time, actor, action and details (written in one fixed, sorted format so the same entry always gives the same fingerprint).
- `append_event` adds an entry. It takes a database lock so two writers cannot link to the same previous entry, and the first entry links to a fixed starting value of 64 zeros.
- `verify_chain` walks every entry in order and reports the first one whose link or content does not match.

In [2]:
%%writefile /home/linux1/bank-rag-guard/src/audit.py
"""Tamper-evident audit log: each row stores a SHA-256 fingerprint that includes the previous row's fingerprint."""
import hashlib, json
from datetime import datetime, timezone
from psycopg.types.json import Jsonb

GENESIS = "0" * 64                                   # the "previous fingerprint" of the very first entry

def canonical(obj):
    """One fixed text form for any JSON value (sorted keys, no extra spaces), so hashes are repeatable."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def ts_text(ts):
    """Timestamp as UTC text with microseconds, the same way every time."""
    if ts.tzinfo is None:                            # a time without a zone is treated as UTC
        ts = ts.replace(tzinfo=timezone.utc)
    return ts.astimezone(timezone.utc).isoformat(timespec="microseconds")

def compute_hash(prev_hash, ts, actor, action, details):
    """SHA-256 of (previous fingerprint + this entry's content)."""
    body = canonical({"ts": ts_text(ts), "actor": actor, "action": action, "details": details})
    return hashlib.sha256((prev_hash + body).encode("utf-8")).hexdigest()

def append_event(conn, actor, action, details=None):
    """Add one entry to the chain and return (id, hash). A lock keeps concurrent writers in order."""
    details = details or {}
    ts = datetime.now(timezone.utc)
    with conn.transaction():
        conn.execute("SELECT pg_advisory_xact_lock(748201)")            # one writer at a time
        row = conn.execute("SELECT hash FROM audit_log ORDER BY id DESC LIMIT 1").fetchone()
        prev = row[0] if row else GENESIS                                # link to the latest entry
        digest = compute_hash(prev, ts, actor, action, details)
        new_id = conn.execute(
            "INSERT INTO audit_log (ts, actor, action, details, prev_hash, hash) "
            "VALUES (%s, %s, %s, %s, %s, %s) RETURNING id",
            (ts, actor, action, Jsonb(details), prev, digest)).fetchone()[0]
    return new_id, digest

def verify_chain(conn):
    """Check every entry in order. Returns (ok, first_bad_id, message)."""
    prev, count = GENESIS, 0
    rows = conn.execute("SELECT id, ts, actor, action, details, prev_hash, hash FROM audit_log ORDER BY id")
    for id_, ts, actor, action, details, prev_hash, digest in rows:
        if prev_hash != prev:                        # the link to the previous entry is broken
            return False, id_, "link to the previous entry is broken (an earlier entry was changed, removed or reordered)"
        if compute_hash(prev_hash, ts, actor, action, details) != digest:
            return False, id_, "this entry's content does not match its fingerprint (it was edited)"
        prev, count = digest, count + 1
    return True, None, f"all {count} entries verified"

Writing /home/linux1/bank-rag-guard/src/audit.py


## 3. Add sample events
Writes five realistic audit events (a login, two searches, a masked answer and a blocked request) and shows the table. Each row's `prev_hash` equals the row above's `hash`, which is the chain.

In [3]:
import importlib, audit
importlib.reload(audit)                              # load the latest version of src/audit.py

events = [
    ("teller1",     "login",          {"role": "teller", "tier": 1}),
    ("teller1",     "search",         {"query": "customer dispute", "tier": 1, "chunks_returned": 5}),
    ("compliance1", "search",         {"query": "suspicious transactions", "tier": 3, "chunks_returned": 5}),
    ("teller1",     "answer_masked",  {"pii_masked": {"NAME": 2, "IBAN": 1}}),
    ("auditor1",    "request_blocked", {"reason": "tier too low for requested document"}),
]
ids = [audit.append_event(conn, a, act, d)[0] for a, act, d in events]

log = pd.DataFrame(conn.execute(
    "SELECT id, actor, action, left(prev_hash, 10) AS prev_hash, left(hash, 10) AS hash FROM audit_log ORDER BY id"
).fetchall(), columns=["id", "actor", "action", "prev_hash (first 10)", "hash (first 10)"])
display(log.style.hide(axis="index"))

id,actor,action,prev_hash (first 10),hash (first 10)
1,teller1,login,0000000000,a3b8fadd12
2,teller1,search,a3b8fadd12,c742f694ea
3,compliance1,search,c742f694ea,cf43666763
4,teller1,answer_masked,cf43666763,365d7df567
5,auditor1,request_blocked,365d7df567,16839d2f5f


## 4. Verify the chain
Recomputes every fingerprint and checks every link. A healthy log reports all entries verified.

In [4]:
ok, bad_id, message = audit.verify_chain(conn)
print("Chain intact:", ok, "|", message)             # expected: True

Chain intact: True | all 5 entries verified


## 5. Make the table append-only
Adds database triggers that reject any UPDATE, DELETE or TRUNCATE on `audit_log`. After this, normal database use can only add new entries. (The next section shows why this alone is not enough.)

In [5]:
# One function that always refuses, attached to three kinds of change
statements = [
    """CREATE OR REPLACE FUNCTION audit_log_block_change() RETURNS trigger AS $$
       BEGIN
         RAISE EXCEPTION 'audit_log is append-only: changes are not allowed';
       END;
       $$ LANGUAGE plpgsql""",
    "DROP TRIGGER IF EXISTS audit_log_no_change ON audit_log",
    """CREATE TRIGGER audit_log_no_change BEFORE UPDATE OR DELETE ON audit_log
       FOR EACH ROW EXECUTE FUNCTION audit_log_block_change()""",
    "DROP TRIGGER IF EXISTS audit_log_no_truncate ON audit_log",
    """CREATE TRIGGER audit_log_no_truncate BEFORE TRUNCATE ON audit_log
       FOR EACH STATEMENT EXECUTE FUNCTION audit_log_block_change()""",
]
for sql in statements:
    conn.execute(sql)

# Try to cheat: each attempt should be refused
for sql in ("UPDATE audit_log SET actor = 'nobody' WHERE id = %s", "DELETE FROM audit_log WHERE id = %s"):
    try:
        conn.execute(sql, (ids[1],))
        print("NOT BLOCKED:", sql)
    except Exception as e:
        print("Blocked ->", str(e).splitlines()[0])

Blocked -> audit_log is append-only: changes are not allowed
Blocked -> audit_log is append-only: changes are not allowed


## 6. Tamper demo: an administrator edits the log
Simulates an insider with full database rights: they switch the trigger off and rewrite one entry's details. The chain check then points to exactly the edited entry. Finally the original value is restored and the trigger switched back on.

In [6]:
target = ids[2]                                      # the compliance search entry
original = conn.execute("SELECT details FROM audit_log WHERE id = %s", (target,)).fetchone()[0]

print("Before tampering :", audit.verify_chain(conn))
try:
    conn.execute("ALTER TABLE audit_log DISABLE TRIGGER audit_log_no_change")     # the insider removes the safeguard
    conn.execute("UPDATE audit_log SET details = %s WHERE id = %s",
                 (Jsonb({"query": "nothing to see here", "tier": 3, "chunks_returned": 0}), target))
    print("After tampering  :", audit.verify_chain(conn))                          # expected: False, at id", target)
finally:
    conn.execute("UPDATE audit_log SET details = %s WHERE id = %s", (Jsonb(original), target))   # undo the edit
    conn.execute("ALTER TABLE audit_log ENABLE TRIGGER audit_log_no_change")                       # safeguard back on
print("After restoring  :", audit.verify_chain(conn))                              # expected: True
print("Tampered entry id was:", target)

Before tampering : (True, None, 'all 5 entries verified')
After tampering  : (False, 3, "this entry's content does not match its fingerprint (it was edited)")
After restoring  : (True, None, 'all 5 entries verified')
Tampered entry id was: 3
